In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import SGD, Adam


In [13]:
def preprocess_emg_signal(file_path, fs = 1000, mvc_dict=None):
    df = pd.read_csv(file_path)
    df_processed = df.copy()
    channels = [f'channel{i}' for i in range(1, 9)]
    nyq = 0.5 * fs
    b_band, a_band = butter(4, [20/nyq, 450/nyq], btype='band')
    b_low, a_low = butter(4, 10/nyq, btype='low')

    for ch in channels:
        if ch not in df.columns:
            print(f"Warning: cannot find {ch} in the file!")
            continue
        raw_sig = df[ch].fillna(0).values
        filtered = filtfilt(b_band, a_band, raw_sig)
        rectified = np.abs(filtered)
        envelope = filtfilt(b_low, a_low, rectified)
        envelope = np.maximum(envelope, 0)
        if mvc_dict is not None and ch in mvc_dict:
            mvc = mvc_dict[ch]
        else:
            mvc = np.max(envelope)
            if mvc == 0: mvc = 1

        normalized = (envelope / mvc) * 100

        df_processed[f'{ch}_norm'] = normalized

        print(df_processed)

    output_file = file_path.replace('.csv', '_processed.csv')
    df_processed.to_csv(output_file, index=False)

    print("Finishing Preprocessing dataset")

    return df_processed


In [14]:
file_path = r"datasets\train\EMG-data.csv"
df = preprocess_emg_signal(file_path)
df 

          time  channel1  channel2  channel3  channel4  channel5  channel6  \
0            1   0.00001  -0.00002  -0.00001  -0.00003   0.00000  -0.00001   
1            5   0.00001  -0.00002  -0.00001  -0.00003   0.00000  -0.00001   
2            6  -0.00001   0.00001   0.00002   0.00000   0.00001  -0.00002   
3            7  -0.00001   0.00001   0.00002   0.00000   0.00001  -0.00002   
4            8  -0.00001   0.00001   0.00002   0.00000   0.00001  -0.00002   
...        ...       ...       ...       ...       ...       ...       ...   
4237902  50962   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237903  50963   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237904  50964   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237905  50965   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   
4237906  50966   0.00001  -0.00001  -0.00002  -0.00004  -0.00012   0.00000   

         channel7  channel8  class  label  channel1_norm  
0   

,time,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class,label,channel1_norm,channel2_norm,channel3_norm,channel4_norm,channel5_norm,channel6_norm,channel7_norm,channel8_norm
0,1,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0,1,0.305582,0.346350,0.000000,0.405888,0.000000,0.054135,0.017472,0.000000
1,5,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0,1,0.326472,0.423523,0.051696,0.532532,0.000000,0.113909,0.032576,0.000000
2,6,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1,0.347065,0.502241,0.201114,0.658244,0.000000,0.172636,0.048004,0.000000
3,7,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1,0.367312,0.582400,0.349487,0.782807,0.000000,0.230206,0.063736,0.000000
4,8,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1,0.387167,0.663891,0.496581,0.906009,0.000000,0.286517,0.079752,0.004179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4237902,50962,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.218362,0.569168,0.896970,0.841954,3.641126,3.038713,1.596788,2.066380
4237903,50963,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.216188,0.569136,0.896311,0.842641,3.644920,3.037181,1.598525,2.066052
4237904,50964,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.214376,0.569113,0.895760,0.843211,3.648063,3.035916,1.599968,2.065764
4237905,50965,0.00001,-0.00001,-0.00002,-0.00004,-0.00012,0.00000,0.00002,0.00002,0,36,1.212884,0.569095,0.895304,0.843679,3.650631,3.034884,1.601151,2.065515


In [15]:
def create_sliding_windows(data, window_size, step_size, channels):
    X, y = [], []
    for (subj, cls_val), group in data.groupby(['label', 'class']):
        group_vals = group[channels].values

        for i in range(0, len(group_vals) - window_size + 1, step_size):
            window = group_vals[i : i + window_size]
            X.append(window)
            y.append(cls_val)
            
    return np.array(X), np.array(y)

In [16]:
def prepare_emg_data_for_cnn(csv_file, window_size=200, step_size=50):
    df = pd.read_csv(csv_file)
    df = df[df['class'] != 0]
    df['class'] = df['class'] - 1

    train_subjects = list(range(1, 31))
    test_subjects = list(range(31, 37))

    train_df = df[df['label'].isin(train_subjects)]
    test_df = df[df['label'].isin(test_subjects)]

    channels = [f'channel{i}' for i in range(1, 9)]

    print("Slicing Training set")
    X_train, y_train = create_sliding_windows(train_df, window_size, step_size, channels)

    print("Slicing Testing set")
    X_test, y_test = create_sliding_windows(test_df, window_size, step_size, channels)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape:  {X_test.shape}")
    print(f"y_test shape:  {y_test.shape}")

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = prepare_emg_data_for_cnn('datasets/train/EMG-data_processed.csv', window_size=200, step_size=50)

Slicing Training set
Slicing Testing set
X_train shape: (24726, 200, 8)
y_train shape: (24726,)
X_test shape:  (4769, 200, 8)
y_test shape:  (4769,)


In [ ]:
def build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=7):
    input_shape = (X_train.shape[1], X_train.shape[2])
    model = Sequential([
        Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=64, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

    print("============== Model Summary: ==============")
    model.summary()

    print("Starting Training......")

    history = model.fit(X_train, y_train, 
                        epochs=30,       
                        batch_size=64,      
                        validation_data=(X_test, y_test))
    return model, history

model, history = build_and_train_cnn(X_train, y_train, X_test, y_test, num_classes=7)

============== Model Summary: ==============


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 198, 32)        │           800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 198, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 99, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 97, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 97, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       393,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401,510 (1.53 MB)

 Trainable params: 401,318 (1.53 MB)

 Non-trainable params: 192 (768.00 B)

Starting Training......
Epoch 1/30
  1/387 ━━━━━━━━━━━━━━━━━━━━ 8:32 1s/step - accuracy: 0.1719 - loss: 1.7835

InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>

  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance

  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start

  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start

  File "c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\asyncio\base_events.py", line 638, in run_forever

  File "c:\Users\Admin\AppDat